In [17]:
import asyncio
import csv
import json
import os
import random
import sys
import time
from datetime import datetime
from pathlib import Path

import getpass
import pandas as pd
from playwright.sync_api import sync_playwright
from playwright_stealth import Stealth
from dotenv import load_dotenv, find_dotenv

In [20]:
# ============================================================
# KONFIGURASI
# ============================================================

# ------------------------------------------------------------
# FILE INPUT (CSV pemisah titik koma ;)
# ------------------------------------------------------------
INPUT_PATH = Path("../list_ub_um_sumsel.csv").resolve()
INPUT_SEP = ";"
KOLOM_ID = "assignment_id"
KOLOM_FLAG = "flag_usaha"          # isinya UB / UM


# ------------------------------------------------------------
# URL & LOGIN
# ------------------------------------------------------------
LINK_BASE_URL = "https://fasih-dashboard.bps.go.id/"
LINK_SQLLAB_URL = "https://fasih-dashboard.bps.go.id/superset/sqllab/"
URL_SQL_EXECUTE = "https://fasih-dashboard.bps.go.id/api/v1/sqllab/execute/"
_env_path = find_dotenv(usecwd=True)
load_dotenv(_env_path, override=True)
print(".env dipakai:", _env_path or "TIDAK DITEMUKAN")

SQLLAB_USERNAME = os.environ.get("SQLLAB_USERNAME")
SQLLAB_PASSWORD = os.environ.get("SQLLAB_PASSWORD")

if not SQLLAB_USERNAME or not SQLLAB_PASSWORD:
    raise RuntimeError(
        "SQLLAB_USERNAME / SQLLAB_PASSWORD kosong. "
        "Cek nama variabel di .env dan lokasi file .env."
    )

SELECTOR_USERNAME = "xpath=//*[@id='username']"
SELECTOR_PASSWORD = "xpath=//*[@id='password']"
SELECTOR_TOMBOL_LOGIN = "xpath=//*[@id='container']/div[2]/form/button"


# ------------------------------------------------------------
# SQL LAB
# ------------------------------------------------------------
DATABASE_ID = 25
SQL_EDITOR_ID = "1036293"
TAB_NAME = "Tagging level_6_full_code"

# Schema & kolom alamat per flag usaha
CONFIG_FLAG = {
    "UB": {"schema": "tcz_37526b20", "kolom_sumber": "alamat_usaha"},
    "UM": {"schema": "tgr_fd68e454", "kolom_sumber": "alamat_prelist"},
}

CHUNK_SIZE = 500
QUERY_LIMIT = 9000

# Kolom output CSV (pemisah koma)
KOLOM_OUTPUT = ["assignment_id", "alamat"]

MAX_RETRY = 5
REQUEST_TIMEOUT = 180000  # ms


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------
OUTPUT_DIR = Path("hasil_scraping").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# BACA INPUT & BAGI PER FLAG
# ============================================================

def baca_assignment_id():
    """
    Return dict {flag: [assignment_id, ...]} (unik, urutan asli dipertahankan).
    """
    df = pd.read_csv(INPUT_PATH, sep=INPUT_SEP, dtype=str, encoding="utf-8-sig")
    df.columns = [str(c).strip() for c in df.columns]

    for k in (KOLOM_ID, KOLOM_FLAG):
        if k not in df.columns:
            raise KeyError(f"Kolom '{k}' tidak ditemukan. Kolom tersedia: {list(df.columns)}")

    total_baris = len(df)
    df = df[[KOLOM_ID, KOLOM_FLAG]].dropna()
    df[KOLOM_ID] = df[KOLOM_ID].str.strip()
    df[KOLOM_FLAG] = df[KOLOM_FLAG].str.strip().str.upper()
    df = df[df[KOLOM_ID] != ""]
    kosong = total_baris - len(df)

    flag_asing = set(df[KOLOM_FLAG]) - set(CONFIG_FLAG)
    if flag_asing:
        raise ValueError(f"Flag tidak dikenal: {flag_asing}. Tambahkan ke CONFIG_FLAG.")

    total_awal = len(df)
    df = df.drop_duplicates(subset=KOLOM_ID, keep="first")

    print("=" * 70)
    print("INPUT CSV")
    print("=" * 70)
    print(f"File             : {INPUT_PATH}")
    print(f"Baris total      : {total_baris:,}")
    print(f"ID/flag kosong   : {kosong:,}")
    print(f"Duplikat dibuang : {total_awal - len(df):,}")
    ids_per_flag = {}
    for flag in CONFIG_FLAG:
        ids_per_flag[flag] = df.loc[df[KOLOM_FLAG] == flag, KOLOM_ID].tolist()
        print(f"  {flag}: {len(ids_per_flag[flag]):,} ID  -> schema {CONFIG_FLAG[flag]['schema']}")
    return ids_per_flag


def buat_jobs(ids_per_flag, ukuran=CHUNK_SIZE):
    """Satu job = satu query = (flag, maks `ukuran` ID). Flag tidak pernah dicampur."""
    jobs = []
    for flag, ids in ids_per_flag.items():
        for i in range(0, len(ids), ukuran):
            jobs.append({"flag": flag, "ids": ids[i:i + ukuran]})
    return jobs


IDS_PER_FLAG = baca_assignment_id()
ASSIGNMENT_IDS = [i for ids in IDS_PER_FLAG.values() for i in ids]
JOBS = buat_jobs(IDS_PER_FLAG)

print(f"Jumlah query     : {len(JOBS):,} (maks {CHUNK_SIZE:,} ID per query)")

.env dipakai: d:\2026\Sensus Ekonomi\Bot Scraper\.env
INPUT CSV
File             : D:\2026\Sensus Ekonomi\list_ub_um_sumsel.csv
Baris total      : 4,856
ID/flag kosong   : 0
Duplikat dibuang : 0
  UB: 257 ID  -> schema tcz_37526b20
  UM: 4,599 ID  -> schema tgr_fd68e454
Jumlah query     : 11 (maks 500 ID per query)


In [21]:
# ============================================================
# MEMBUAT SQL (MINIFIED, 1 BARIS) - BEDA SCHEMA & KOLOM PER FLAG
# ============================================================

def buat_sql(flag, chunk):
    """
    UB: SELECT assignment_id, alamat_usaha   FROM tcz_37526b20.root_table WHERE assignment_id IN (...)
    UM: SELECT assignment_id, alamat_prelist FROM tgr_fd68e454.root_table WHERE assignment_id IN (...)
    """
    cfg = CONFIG_FLAG[flag]
    daftar = ",".join("'" + i.replace("'", "''") + "'" for i in chunk)

    sql = (
        f"SELECT assignment_id, {cfg['kolom_sumber']} "
        f"FROM {cfg['schema']}.root_table "
        f"WHERE assignment_id IN ({daftar})"
    )

    assert "\n" not in sql, "SQL tidak boleh mengandung baris baru"
    return sql


def petakan_rows(rows, flag):
    """Samakan nama kolom hasil (alamat_usaha / alamat_prelist) menjadi 'alamat'."""
    kolom = CONFIG_FLAG[flag]["kolom_sumber"]
    return [
        {"assignment_id": r.get("assignment_id"), "alamat": r.get(kolom)}
        for r in rows
        if isinstance(r, dict)
    ]


# Contoh SQL (3 ID pertama tiap flag, untuk pengecekan bentuk)
for _flag, _ids in IDS_PER_FLAG.items():
    if _ids:
        print(f"[{_flag}]", buat_sql(_flag, _ids[:3]), "\n")


[UB] SELECT assignment_id, alamat_usaha FROM tcz_37526b20.root_table WHERE assignment_id IN ('021cbbac-58f8-42fc-94bf-2f4226f530f8','58a31b1c-87a2-460d-b53c-45c9b6b9b8be','867bc87b-d05a-4bc4-b47b-66df99df695e') 

[UM] SELECT assignment_id, alamat_prelist FROM tgr_fd68e454.root_table WHERE assignment_id IN ('f5933158-a515-4874-88d2-8a43c13fe6eb','75126543-e0cb-419c-be86-2195e62e8ef1','61ebf9d6-7964-4b82-bea0-3ef40cdc5560') 



In [22]:
# ============================================================
# FIX EVENT LOOP UNTUK PLAYWRIGHT DI JUPYTER (WINDOWS)
# ============================================================

class GunakanProactorEventLoop:
    """
    Memaksa Windows memakai ProactorEventLoop selama Playwright aktif,
    lalu mengembalikan policy semula setelah selesai.
    """

    def __enter__(self):
        self.policy_lama = None
        if sys.platform == "win32":
            self.policy_lama = asyncio.get_event_loop_policy()
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        return self

    def __exit__(self, exc_type, exc_value, tb):
        if self.policy_lama is not None:
            asyncio.set_event_loop_policy(self.policy_lama)


# ============================================================
# LOGIN
# ============================================================

def isi_username_password(page):
    page.wait_for_selector(SELECTOR_USERNAME, state="visible", timeout=15000)
    # username = input("Username SQL Lab: ").strip()
    # password = getpass.getpass("Password SQL Lab: ").strip()
    page.fill(SELECTOR_USERNAME, SQLLAB_USERNAME)
    page.fill(SELECTOR_PASSWORD, SQLLAB_PASSWORD)
    page.press(SELECTOR_PASSWORD, "Enter")
    print("Username & password terisi, form disubmit.")


def klik_tombol_login(page, max_percobaan=5, timeout_ms=15000):
    try:
        page.wait_for_selector(
            SELECTOR_TOMBOL_LOGIN, timeout=timeout_ms, state="visible"
        )
    except Exception:
        print("Tombol login tidak ditemukan.")
        print("Kemungkinan browser sudah login atau butuh autentikasi tambahan.")
        return False

    url_sebelum = page.url
    for percobaan in range(1, max_percobaan + 1):
        try:
            time.sleep(random.uniform(1, 3))
            page.locator(SELECTOR_TOMBOL_LOGIN).click(timeout=5000)
        except Exception as e:
            print(f"Percobaan {percobaan} - klik login gagal: {e}")
            page.wait_for_timeout(1500)
            continue

        page.wait_for_timeout(2000)
        ada_perubahan_url = page.url != url_sebelum
        ada_form_username = page.locator(SELECTOR_USERNAME).count() > 0

        if ada_perubahan_url or ada_form_username:
            print(f"Tombol login berhasil diproses (percobaan {percobaan}).")
            if ada_form_username:
                isi_username_password(page)
            return True
        print(f"Percobaan {percobaan} belum ada perubahan, mencoba lagi...")

    return False


def login_dashboard(page):
    print("\n" + "=" * 70)
    print("LOGIN FASIH DASHBOARD")
    print("=" * 70)
    page.goto(LINK_BASE_URL, wait_until="domcontentloaded")
    login_selesai = klik_tombol_login(page)
    if not login_selesai:
        print("\nSilakan selesaikan login/captcha jika diperlukan.")
        input("Setelah login berhasil, tekan ENTER untuk melanjutkan...")
        if page.locator(SELECTOR_USERNAME).count() > 0:
            isi_username_password(page)
    page.wait_for_timeout(3000)
    print("\nLogin selesai.")
    print("URL:", page.url)


# ============================================================
# MENANGKAP CSRF DARI REQUEST SQL LAB
# ============================================================

def pasang_penangkap_csrf(page):
    request_info = {"csrf_token": None}

    def handle_request(request):
        if "/api/v1/sqllab/execute/" not in request.url:
            return
        csrf_token = request.headers.get("x-csrftoken")
        if not csrf_token:
            return
        request_info["csrf_token"] = csrf_token
        print("\n[OK] CSRF token berhasil ditangkap.")

    page.on("request", handle_request)
    return request_info


def tunggu_csrf(page, request_info, timeout_detik=300):
    print("\n" + "=" * 70)
    print("MENUNGGU REQUEST SQL LAB")
    print("=" * 70)
    print(
        "SQL Lab sudah dibuka.\n"
        "Silakan jalankan SATU query manual (tombol RUN), contoh: SELECT 1\n"
        "Token akan ditangkap otomatis. Jangan tampilkan/kirim nilai token."
    )

    batas_waktu = time.time() + timeout_detik
    while time.time() < batas_waktu:
        if request_info["csrf_token"]:
            print("\n[OK] CSRF token sudah tersedia.")
            return request_info["csrf_token"]
        page.wait_for_timeout(500)

    raise Exception(
        "Timeout menunggu request SQL Lab. "
        "Pastikan tombol RUN sudah ditekan dan query berhasil dijalankan."
    )


# ============================================================
# PAYLOAD & EKSEKUSI SQL
# ============================================================

def buat_payload(sql, schema):
    return {
        "database_id": DATABASE_ID,
        "json": True,
        "runAsync": False,
        "schema": schema,
        "sql": sql,
        "sql_editor_id": SQL_EDITOR_ID,
        "tab": TAB_NAME,
        "tmp_table_name": "",
        "select_as_cta": False,
        "ctas_method": "TABLE",
        "queryLimit": QUERY_LIMIT,
        "expand_data": True,
    }


def execute_query(context, sql, csrf_token, label, schema):
    """
    Menjalankan SQL memakai session browser yang sudah login.
    Return: dict hasil | "REFRESH_CSRF" | None
    """
    payload = buat_payload(sql, schema)
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "X-CSRFToken": csrf_token,
        "Referer": LINK_SQLLAB_URL,
    }

    for percobaan in range(1, MAX_RETRY + 1):
        print("\n" + "-" * 70)
        print(f"{label} | Percobaan {percobaan}/{MAX_RETRY}")
        try:
            response = context.request.post(
                URL_SQL_EXECUTE, data=payload, headers=headers, timeout=REQUEST_TIMEOUT
            )
            print("HTTP Status:", response.status)
            response_text = response.text()

            if response.status == 200:
                try:
                    result = response.json()
                    print("[OK] Query berhasil.")
                    return result
                except Exception as e:
                    print("Response 200 tetapi bukan JSON:", e)
                    print(response_text[:500])
                    return None

            if response.status == 400:
                print(response_text[:500])
                if "csrf" in response_text.lower():
                    print("[WARNING] CSRF token tidak valid.")
                    return "REFRESH_CSRF"

            elif response.status in (401, 403):
                print("Session kemungkinan sudah expired.")
                print(response_text[:500])
                return None

            else:
                print("Request gagal.")
                print(response_text[:500])

        except Exception as e:
            print(f"Request error: {e}")

        if percobaan < MAX_RETRY:
            waktu_tunggu = min(5 * percobaan, 30)
            print(f"Menunggu {waktu_tunggu} detik...")
            time.sleep(waktu_tunggu)

    print(f"\n[GAGAL] {label}")
    return None


def ambil_rows(result):
    """Mengambil list row dari beberapa kemungkinan struktur response Superset."""
    if not isinstance(result, dict):
        return []

    data = result.get("data")
    if isinstance(data, list):
        return data

    result_data = result.get("result")
    if isinstance(result_data, list):
        for item in result_data:
            if isinstance(item, dict) and isinstance(item.get("data"), list):
                return item["data"]
    if isinstance(result_data, dict) and isinstance(result_data.get("data"), list):
        return result_data["data"]

    return []

In [23]:
# ============================================================
# STATE RESUME (PER JOB)
# ============================================================

def buat_state_baru(nama_file):
    return {
        "nama_file": str(nama_file),
        "input": str(INPUT_PATH),
        "total_id": len(ASSIGNMENT_IDS),
        "chunk_size": CHUNK_SIZE,
        "total_chunk": len(JOBS),
        "next_chunk": 0,          # index job berikutnya yang akan dijalankan
        "total_data": 0,
        "header_sudah_ditulis": False,
        "file_size": 0,
        "selesai": False,
        "updated_at": datetime.now().isoformat(),
    }


def baca_state(state_file):
    if not state_file.exists():
        return None
    try:
        with open(state_file, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"[WARNING] State tidak dapat dibaca: {e}")
        return None


def simpan_state(state_file, state):
    """Simpan atomic: tulis ke file temp lalu replace."""
    temp_file = state_file.with_suffix(".tmp")
    state["updated_at"] = datetime.now().isoformat()
    with open(temp_file, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(temp_file, state_file)


def state_cocok_dengan_input(state):
    """State hanya dipakai ulang jika input & pengaturan chunk sama."""
    return (
        state.get("input") == str(INPUT_PATH)
        and state.get("total_id") == len(ASSIGNMENT_IDS)
        and state.get("chunk_size") == CHUNK_SIZE
        and state.get("total_chunk") == len(JOBS)
    )


def siapkan_output():
    state_files = sorted(
        OUTPUT_DIR.glob("hasil_alamat_*.state.json"), key=lambda x: x.stat().st_mtime, reverse=True
    )

    for state_file in state_files:
        state = baca_state(state_file)
        if not state or not state_cocok_dengan_input(state):
            continue

        nama_file = Path(state["nama_file"])
        if not nama_file.is_absolute():
            nama_file = OUTPUT_DIR / nama_file

        print("\n" + "=" * 70)
        print("SELESAI SEBELUMNYA" if state.get("selesai") else "RESUME SCRAPING")
        print("=" * 70)
        print("File CSV   :", nama_file)
        print(f"Chunk      : {state['next_chunk']:,} / {state['total_chunk']:,} selesai")
        print(f"Total data : {state['total_data']:,}")
        return nama_file, state_file, state

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    nama_file = OUTPUT_DIR / f"hasil_alamat_{timestamp}.csv"
    state_file = OUTPUT_DIR / f"hasil_alamat_{timestamp}.state.json"
    state = buat_state_baru(nama_file)
    simpan_state(state_file, state)

    print("\n" + "=" * 70)
    print("SCRAPING BARU")
    print("=" * 70)
    print("File CSV :", nama_file)
    print("State    :", state_file)
    return nama_file, state_file, state


def pulihkan_file_csv(nama_file, state):
    """Potong CSV ke ukuran di state agar tidak ada batch ganda setelah crash."""
    file_size = state.get("file_size", 0)
    if not nama_file.exists():
        return
    if nama_file.stat().st_size > file_size:
        print("\n[RESUME] Memulihkan ukuran CSV...")
        with open(nama_file, "r+b") as f:
            f.truncate(file_size)
        print("[OK] CSV berhasil dipulihkan.")


def simpan_batch_csv(rows, nama_file, header_sudah_ditulis):
    """Tulis satu batch ke CSV (pemisah KOMA). Return (header_sudah_ditulis, file_size)."""
    if not rows:
        return header_sudah_ditulis, (nama_file.stat().st_size if nama_file.exists() else 0)

    mode = "a" if header_sudah_ditulis else "w"

    with open(nama_file, mode, newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=KOLOM_OUTPUT,
            delimiter=",",
            quoting=csv.QUOTE_MINIMAL,   # alamat berisi koma otomatis di-quote
            extrasaction="ignore",
            restval="",
        )
        if not header_sudah_ditulis:
            writer.writeheader()
        for row in rows:
            writer.writerow(row)
        f.flush()
        os.fsync(f.fileno())

    return True, nama_file.stat().st_size


In [24]:
# ============================================================
# SCRAP SEMUA CHUNK
# ============================================================

def scrap_semua_data(context, csrf_token, nama_file, state_file, state):
    pulihkan_file_csv(nama_file, state)

    total_chunk = len(JOBS)
    mulai = state.get("next_chunk", 0)
    total_data = state.get("total_data", 0)
    header_sudah_ditulis = state.get("header_sudah_ditulis", False)

    print("\n" + "=" * 70)
    print("MEMULAI SCRAPING PER CHUNK ASSIGNMENT_ID")
    print("=" * 70)
    print(f"Mulai dari chunk : {mulai + 1:,} / {total_chunk:,}")
    print(f"Total data awal  : {total_data:,}")
    print("=" * 70)

    for idx in range(mulai, total_chunk):
        job = JOBS[idx]
        flag, chunk = job["flag"], job["ids"]
        label = f"CHUNK {idx + 1:,}/{total_chunk:,} | {flag} | {len(chunk):,} ID"

        result = execute_query(
            context=context,
            sql=buat_sql(flag, chunk),
            csrf_token=csrf_token,
            label=label,
            schema=CONFIG_FLAG[flag]["schema"],
        )

        if result == "REFRESH_CSRF" or result is None:
            print(f"\n[ERROR] {label} gagal. Scraping dihentikan.")
            print("Jalankan notebook lagi untuk melanjutkan dari chunk ini.")
            return nama_file, total_data, False

        rows = petakan_rows(ambil_rows(result), flag)
        jumlah = len(rows)
        print(f"Row diterima : {jumlah:,} (ID dikirim: {len(chunk):,})")

        if jumlah == QUERY_LIMIT and len(chunk) == QUERY_LIMIT:
            print("[INFO] Row tepat di batas limit; cek jika ada ID dengan banyak baris di root_table.")
        if jumlah < len(chunk):
            print(f"[INFO] {len(chunk) - jumlah:,} ID tidak ditemukan di root_table.")

        header_sudah_ditulis, file_size = simpan_batch_csv(
            rows=rows, nama_file=nama_file, header_sudah_ditulis=header_sudah_ditulis
        )
        total_data += jumlah

        # State disimpan SETELAH data benar-benar masuk ke disk
        state["next_chunk"] = idx + 1
        state["total_data"] = total_data
        state["header_sudah_ditulis"] = header_sudah_ditulis
        state["file_size"] = file_size
        state["selesai"] = (idx + 1 >= total_chunk)
        simpan_state(state_file, state)

        print(f"Total data   : {total_data:,}")
        print(f"Ukuran CSV   : {file_size / (1024 * 1024):,.2f} MB")

        if idx + 1 < total_chunk:
            jeda = random.uniform(1, 3)
            print(f"Menunggu {jeda:.1f} detik...")
            time.sleep(jeda)

    return nama_file, total_data, True


# ============================================================
# MAIN SCRAPING
# ============================================================

def jalankan_scraping():
    nama_file, state_file, state = siapkan_output()

    if state.get("selesai"):
        print("\nScraping sebelumnya sudah selesai.")
        print("Hapus file *.state.json di folder hasil_scraping untuk mengulang dari awal.")
        return nama_file, state.get("total_data", 0)

    with GunakanProactorEventLoop():
        with sync_playwright() as p:
            browser = p.chromium.launch(
                headless=False, args=["--disable-blink-features=AutomationControlled"]
            )
            context = browser.new_context(
                accept_downloads=True,
                user_agent=(
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/126.0.0.0 Safari/537.36"
                ),
                viewport={"width": 1366, "height": 768},
                locale="id-ID",
                timezone_id="Asia/Jakarta",
            )
            page = context.new_page()
            Stealth().apply_stealth_sync(page)

            try:
                login_dashboard(page)
                request_info = pasang_penangkap_csrf(page)

                print("\nMembuka SQL Lab...")
                page.goto(LINK_SQLLAB_URL, wait_until="domcontentloaded")
                page.wait_for_timeout(5000)

                csrf_token = tunggu_csrf(page, request_info)

                print("\nMemulai scraping otomatis...")
                nama_file, total_data, selesai = scrap_semua_data(
                    context=context,
                    csrf_token=csrf_token,
                    nama_file=nama_file,
                    state_file=state_file,
                    state=state,
                )

                print("\n" + "=" * 70)
                print("SCRAPING SELESAI" if selesai else "SCRAPING BERHENTI / BELUM SELESAI")
                print("=" * 70)
                print(f"Total data : {total_data:,}")
                print(f"File CSV   : {nama_file.resolve()}")
                return nama_file, total_data

            finally:
                print("\nMenutup browser...")
                context.close()
                browser.close()


In [25]:
# ============================================================
# MAIN
# ============================================================

print("\n" + "=" * 70)
print("BOT SQL LAB FASIH - AMBIL ALAMAT (UB / UM) DARI assignment_id")
print("=" * 70)

nama_file, total_data = await asyncio.to_thread(jalankan_scraping)

print("\n" + "=" * 70)
print("SELESAI")
print("=" * 70)
print(f"Total data : {total_data:,}")
print(f"File CSV   : {Path(nama_file).resolve()}")



BOT SQL LAB FASIH - AMBIL ALAMAT (UB / UM) DARI assignment_id

RESUME SCRAPING
File CSV   : D:\2026\Sensus Ekonomi\Bot Scraper\hasil_scraping\hasil_alamat_20261012_090531.csv
Chunk      : 0 / 11 selesai
Total data : 0

LOGIN FASIH DASHBOARD
Tombol login berhasil diproses (percobaan 1).
Username & password terisi, form disubmit.

Login selesai.
URL: https://fasih-dashboard.bps.go.id/superset/welcome/

Membuka SQL Lab...

MENUNGGU REQUEST SQL LAB
SQL Lab sudah dibuka.
Silakan jalankan SATU query manual (tombol RUN), contoh: SELECT 1
Token akan ditangkap otomatis. Jangan tampilkan/kirim nilai token.

[OK] CSRF token sudah tersedia.

Memulai scraping otomatis...

MEMULAI SCRAPING PER CHUNK ASSIGNMENT_ID
Mulai dari chunk : 1 / 11
Total data awal  : 0

----------------------------------------------------------------------
CHUNK 1/11 | UB | 257 ID | Percobaan 1/5
HTTP Status: 200
[OK] Query berhasil.
Row diterima : 257 (ID dikirim: 257)
Total data   : 257
Ukuran CSV   : 0.01 MB
Menunggu 1.5 

In [26]:
# ============================================================
# VALIDASI HASIL: ID YANG TIDAK DITEMUKAN + CEK FORMAT CSV
# ============================================================

df_hasil = pd.read_csv(nama_file, sep=",", dtype=str, encoding="utf-8-sig")
print("Kolom CSV      :", list(df_hasil.columns))
print(f"Baris hasil    : {len(df_hasil):,}")
print(f"Alamat kosong  : {df_hasil['alamat'].isna().sum():,}")

ketemu = set(df_hasil["assignment_id"].str.strip())
df_input = pd.read_csv(INPUT_PATH, sep=INPUT_SEP, dtype=str, encoding="utf-8-sig")
df_input[KOLOM_ID] = df_input[KOLOM_ID].str.strip()
df_hilang = df_input[~df_input[KOLOM_ID].isin(ketemu)][[KOLOM_ID, KOLOM_FLAG]]
print(f"ID tidak ditemukan di root_table: {len(df_hilang):,}")

if len(df_hilang):
    file_missing = OUTPUT_DIR / "assignment_id_tidak_ditemukan.csv"
    df_hilang.to_csv(file_missing, index=False, encoding="utf-8-sig")
    print("Disimpan ke:", file_missing)


Kolom CSV      : ['assignment_id', 'alamat']
Baris hasil    : 4,856
Alamat kosong  : 171
ID tidak ditemukan di root_table: 0


In [31]:
# ============================================================
# ISI KOLOM ALAMAT KE list_ub_um_sumsel.csv DARI HASIL SCRAP
# (tanpa scraping ulang)
# ============================================================
from pathlib import Path
import pandas as pd

INPUT_PATH = Path("../list_ub_um_sumsel.csv").resolve()
INPUT_SEP = ";"  # pemisah file asli
OUTPUT_SEP = (
    ","  # pemisah file hasil gabungan (ganti ";" jika ingin sama seperti file asli)
)
OUTPUT_DIR = Path(r".\hasil_scraping").resolve()

# Pakai CSV hasil scrap terbaru; atau isi manual, contoh:
# FILE_HASIL = OUTPUT_DIR / "hasil_alamat_20261012_153000.csv"
FILE_HASIL = max(OUTPUT_DIR.glob("hasil_alamat_*.csv"), key=lambda p: p.stat().st_mtime)
print("CSV hasil scrap :", FILE_HASIL)

# --- Baca file asli & hasil scrap ---
df_list = pd.read_csv(INPUT_PATH, sep=INPUT_SEP, dtype=str, encoding="utf-8-sig")
df_list.columns = [c.strip() for c in df_list.columns]

df_alamat = pd.read_csv(FILE_HASIL, sep=",", dtype=str, encoding="utf-8-sig")
df_alamat.columns = [c.strip() for c in df_alamat.columns]
df_alamat["assignment_id"] = df_alamat["assignment_id"].str.strip()

# Jika satu assignment_id punya beberapa baris, ambil yang pertama
dobel = df_alamat["assignment_id"].duplicated().sum()
if dobel:
    print(
        f"[INFO] {dobel:,} baris duplikat assignment_id di hasil scrap (diambil yang pertama)."
    )
df_alamat = df_alamat.drop_duplicates(subset="assignment_id", keep="first")[
    ["assignment_id", "alamat"]
]

# --- Gabung (kunci dibersihkan spasi) ---
df_list["_kunci"] = df_list["assignment_id"].str.strip()
df_gabung = (
    df_list.drop(
        columns=["alamat"], errors="ignore"
    )  # buang kolom 'alamat' lama jika ada
    .merge(
        df_alamat.rename(columns={"assignment_id": "_kunci"}), how="left", on="_kunci"
    )
    .drop(columns="_kunci")
)

# Letakkan kolom 'alamat' tepat setelah alamat_usaha
kolom = [c for c in df_gabung.columns if c != "alamat"]
posisi = kolom.index("alamat_usaha") + 1 if "alamat_usaha" in kolom else len(kolom)
kolom.insert(posisi, "alamat")
df_gabung = df_gabung[kolom]

# --- Ringkasan ---
terisi = df_gabung["alamat"].notna().sum()
print(f"Baris list        : {len(df_gabung):,}")
print(f"Alamat terisi     : {terisi:,}")
print(f"Alamat kosong     : {len(df_gabung) - terisi:,}")
print(
    df_gabung.groupby("flag_usaha")["alamat"]
    .apply(lambda s: s.notna().sum())
    .rename("terisi per flag")
)

# --- Simpan ---
file_keluar = INPUT_PATH.with_name(INPUT_PATH.stem + "_dengan_alamat.csv")
df_gabung.to_csv(file_keluar, index=False, sep=OUTPUT_SEP, encoding="utf-8-sig")
print("Disimpan ke:", file_keluar)

CSV hasil scrap : D:\2026\Sensus Ekonomi\Bot Scraper\hasil_scraping\hasil_alamat_20261012_090531.csv
Baris list        : 4,856
Alamat terisi     : 4,685
Alamat kosong     : 171
flag_usaha
UB      86
UM    4599
Name: terisi per flag, dtype: int64
Disimpan ke: D:\2026\Sensus Ekonomi\list_ub_um_sumsel_dengan_alamat.csv
